# Laboratório 5 – Câmera Estéreo
# Visão Computacional — 2026.2

---

**Autores:**
 - João Victor de Castro Carvalho — RA 11202420287
 - Mario Guerreiro Cerconvis — RA 11202421459
 - Marco Antonio Silva da Mata — RA 11202320279
 - João Victor Gonçalves Nascimento — RA 11202420821

**Data de realização dos experimentos:** _[preencher com a data do laboratório]_

**Data de publicação do relatório:** _[preencher]_

---

## 1. Introdução

Este relatório apresenta os estudos e experimentos realizados no Laboratório 5 da disciplina de Visão Computacional, cujo tema central é a **estereoscopia** e a **geometria epipolar**. O objetivo é compreender como um par de câmeras — a **câmera estéreo** — permite recuperar informação de profundidade a partir de duas vistas de uma mesma cena, de forma análoga à visão binocular humana.

Ao longo do laboratório:

1. Estudamos os fundamentos teóricos da geometria epipolar (epipolos, plano epipolar, linhas epipolares, matriz fundamental e disparidade).
2. Construímos uma **câmera estereoscópica didática** com duas webcams USB iguais, montadas paralelamente sobre uma base rígida, com distância entre os eixos ópticos aproximadamente igual à distância inter-pupilar de um integrante da equipe.
3. Executamos o exemplo de calibração estéreo e geração de imagem 3D da referência LearnOpenCV, primeiro com as imagens fornecidas e, em seguida, com as imagens capturadas pela nossa própria câmera.
4. Geramos e gravamos uma **imagem 3D anaglifo** (vermelho/ciano), analisando a percepção de profundidade ao vivo e no vídeo gravado.
5. Discutimos como a câmera estéreo pode ser aplicada no Trabalho Final do grupo.

O relatório está organizado da seguinte forma:
1. **Introdução**
2. **Fundamentação Teórica** — geometria epipolar, matriz fundamental, disparidade e calibração estéreo
3. **Procedimentos Experimentais** — construção da câmera e execução dos códigos de calibração e imagem 3D
4. **Análise e Discussão** — respostas às questões propostas e análise dos parâmetros obtidos
5. **Conclusões**
6. **Referências**

> **Observação sobre as mídias:** os experimentos das Partes 2 e 3 são realizados presencialmente no laboratório, com as webcams USB e o óculos 3D anaglifo. Por isso, ao longo do notebook há **espaços reservados** (marcados como `[ESPAÇO RESERVADO PARA MÍDIA]`) para as imagens de calibração, capturas de tela da imagem 3D e o vídeo gravado, que devem ser inseridos após a aula prática.

## 2. Fundamentação Teórica

### 2.1 Estereoscopia e Visão Binocular

A **estereoscopia** é a técnica de obter a percepção de profundidade (3D) a partir de duas imagens de uma mesma cena, capturadas de pontos de vista ligeiramente diferentes. É exatamente o princípio da visão humana: cada olho vê a cena de uma posição distinta, e o cérebro funde as duas imagens, interpretando as pequenas diferenças entre elas como profundidade.

Em visão computacional, reproduzimos esse mecanismo com **duas câmeras** (ou uma câmera em duas posições). O par de imagens resultante — a imagem da esquerda e a da direita — contém, nas diferenças de posição dos pontos correspondentes, a informação necessária para reconstruir a estrutura 3D da cena.

### 2.2 Geometria Epipolar

A **geometria epipolar** descreve as relações geométricas entre duas vistas de uma cena 3D obtidas por duas câmeras (ou pela mesma câmera em duas posições). Ela é intrínseca: depende apenas dos parâmetros internos das câmeras e da posição relativa entre elas, e **não** da estrutura da cena.

Considere um ponto 3D $X$ observado pelas duas câmeras, com centros ópticos $O_L$ e $O_R$. Os principais elementos da geometria epipolar são:

- **Epipolos ($e_L$ e $e_R$):** o epipolo de uma imagem é a **projeção do centro óptico da outra câmera** naquele plano de imagem. Equivalentemente, é o ponto onde a linha que une os dois centros ópticos (a *linha de base* ou *baseline*) intercepta o plano de imagem. Todas as linhas epipolares de uma imagem passam pelo seu epipolo.

- **Plano epipolar:** é o plano que contém o ponto 3D $X$ e os dois centros ópticos $O_L$ e $O_R$. Para cada ponto 3D da cena existe um plano epipolar; todos eles contêm a linha de base e formam um feixe de planos em torno dela.

- **Linha epipolar:** é a **interseção do plano epipolar com o plano de imagem**. É a imagem, em uma câmera, do raio óptico que parte da outra câmera. Sua importância prática é a **restrição epipolar**: dado um ponto na imagem da esquerda, seu correspondente na imagem da direita **necessariamente** está sobre a linha epipolar correspondente. Isso reduz a busca de correspondências de um problema 2D (a imagem inteira) para um problema 1D (uma única linha), tornando o *matching* estéreo muito mais eficiente e robusto.

### 2.3 Matriz Fundamental e Matriz Essencial

A restrição epipolar é expressa algebricamente pela **matriz fundamental** $F$, uma matriz $3 \times 3$ que relaciona pontos correspondentes entre as duas imagens **em coordenadas de pixel**. Para um ponto $x$ na imagem da esquerda e seu correspondente $x'$ na direita (em coordenadas homogêneas), vale:

$$x'^{\top} F\, x = 0$$

Propriedades e parâmetros da matriz fundamental:

- É uma matriz $3 \times 3$ com **7 graus de liberdade**: possui 9 elementos, mas é definida a menos de um fator de escala (−1 grau) e é **singular**, isto é, $\det(F) = 0$, com posto 2 (−1 grau).
- $F\,x$ fornece a **linha epipolar na imagem da direita** associada ao ponto $x$; analogamente, $F^{\top} x'$ dá a linha epipolar na imagem da esquerda.
- Os **epipolos** são os vetores do espaço nulo de $F$: $F\,e_L = 0$ e $F^{\top} e_R = 0$.
- Pode ser estimada a partir de correspondências de pontos (por exemplo, com o **algoritmo dos 8 pontos**) sem conhecer os parâmetros das câmeras.

Quando as câmeras estão **calibradas** (matrizes intrínsecas $K_L$ e $K_R$ conhecidas), trabalha-se com a **matriz essencial** $E$, que relaciona pontos em coordenadas normalizadas:

$$E = K_R^{\top} F\, K_L, \qquad E = [t]_\times R$$

onde $R$ e $t$ são a rotação e a translação entre as duas câmeras. A matriz essencial tem **5 graus de liberdade** (3 da rotação + 2 da translação, que também é definida a menos de escala) e permite recuperar diretamente a pose relativa entre as câmeras.

### 2.4 Disparidade Estéreo

A **disparidade estéreo** é a diferença de posição horizontal de um mesmo ponto da cena entre a imagem da esquerda e a da direita, em um par estéreo **retificado** (isto é, com as linhas epipolares alinhadas horizontalmente). Se um ponto aparece na coluna $x_L$ na imagem esquerda e $x_R$ na direita, a disparidade é:

$$d = x_L - x_R$$

A disparidade é **inversamente proporcional à profundidade** $Z$ do ponto. Para câmeras paralelas com distância entre eixos ópticos (baseline) $B$ e distância focal $f$:

$$Z = \frac{f \cdot B}{d}$$

Ou seja:
- Objetos **próximos** produzem **grande disparidade** (deslocam-se muito entre as duas vistas).
- Objetos **distantes** produzem **pequena disparidade** (quase não se deslocam).

O mapa de disparidade — imagem em que cada pixel guarda o valor de $d$ — é, portanto, uma representação direta da profundidade da cena, e é a base da reconstrução 3D estéreo. A retificação estéreo é a etapa que transforma o par de imagens de modo que as linhas epipolares fiquem horizontais e alinhadas, reduzindo a busca de disparidade a uma varredura ao longo das linhas.

### 2.5 Calibração da Câmera Estéreo

Para medir profundidade de forma métrica é necessário **calibrar** a câmera estéreo. A calibração determina:

- **Parâmetros intrínsecos** de cada câmera: matriz $K$ (distância focal $f_x, f_y$ e centro óptico $c_x, c_y$) e coeficientes de **distorção da lente** (radial $k_1, k_2, k_3$ e tangencial $p_1, p_2$).
- **Parâmetros extrínsecos (estéreo):** a rotação $R$ e a translação $t$ entre as duas câmeras.
- **Matriz essencial** $E$ e **matriz fundamental** $F$.
- **Matrizes de retificação** e a **matriz de reprojeção** $Q$ (que converte disparidade em coordenadas 3D).

O procedimento usa múltiplas imagens de um **padrão de calibração** conhecido (tabuleiro de xadrez) vistas simultaneamente pelas duas câmeras. Detectam-se os cantos do tabuleiro, associam-se os pontos 2D das imagens aos pontos 3D do padrão e resolve-se um problema de otimização (método de Zhang) que minimiza o erro de reprojeção. No OpenCV, isso corresponde às funções `cv2.calibrateCamera`, `cv2.stereoCalibrate` e `cv2.stereoRectify`, utilizadas nos scripts deste laboratório.

## 3. Procedimentos Experimentais

Esta seção descreve, com o máximo de detalhes para permitir a reprodução, os procedimentos das Partes 2 e 3 do laboratório.

### 3.0 Configuração do Ambiente

As bibliotecas necessárias são `opencv-python`, `numpy`, `matplotlib` e `tqdm`. A célula abaixo importa e verifica as versões.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

get_ipython().run_line_magic('matplotlib', 'inline')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

### 3.1 Parte 2 — Construção da Câmera Estereoscópica

A câmera estereoscópica foi construída com **duas webcams USB iguais** do laboratório, fixadas paralelamente sobre uma base rígida (caixa de sapato / placa de material rígido), seguindo os requisitos da seção *"Steps To Create The Stereo Camera Setup"* da referência [2].

**Procedimento detalhado:**

1. **Alinhamento paralelo:** as duas webcams foram apoiadas sobre uma superfície plana e firme, com os eixos ópticos paralelos e apontando na mesma direção (montagem *fronto-parallel*).
2. **Distância inter-pupilar (baseline):** mediu-se a distância entre as pupilas de um integrante da equipe e utilizou-se esse valor como a distância entre os eixos ópticos das câmeras.
   - Distância inter-pupilar medida: **_[preencher] mm_** (valor típico: 60–65 mm).
3. **Fixação rígida:** após o posicionamento, as câmeras foram **fortemente fixadas** à base, de modo que não se movam uma em relação à outra. Isso é essencial, pois qualquer movimento relativo após a calibração invalida os parâmetros estéreo.
4. **Identificação da câmera:** a câmera recebeu o nome **_[nome da câmera / decoração da equipe]_**.

> **`[ESPAÇO RESERVADO PARA MÍDIA]`** — Inserir aqui foto(s) da câmera estéreo construída pela equipe (vista frontal e superior), destacando a montagem paralela e a base rígida. Salvar em `imagens/camera_estereo.jpg` e exibir com a célula abaixo.

In [ ]:
# Exibição da foto da câmera estéreo construída (inserir após o laboratório).
import os
import matplotlib.image as mpimg

_foto = "imagens/camera_estereo.jpg"
if os.path.exists(_foto):
    plt.figure()
    plt.imshow(mpimg.imread(_foto))
    plt.axis("off")
    plt.title("Câmera estéreo construída pela equipe")
    plt.show()
else:
    print(f"[ESPAÇO RESERVADO] Adicione a foto da câmera em: {_foto}")

### 3.2 Parte 3 (A e B) — Exemplo de Calibração com as Imagens Fornecidas

Os códigos de exemplo foram obtidos do repositório LearnOpenCV (referência [2], pasta `stereo-camera`), que contém três programas em Python e as subpastas de imagens `stereoL` (esquerda) e `stereoR` (direita):

- `capture_images.py` — captura os pares de imagens do tabuleiro;
- `calibrate.py` — realiza a calibração estéreo e salva os parâmetros em `params_py.xml`;
- `movie3d.py` — lê os parâmetros e gera/apresenta a imagem 3D anaglifo.

Com as imagens fornecidas pelo exemplo, executa-se:

```bash
python3 calibrate.py     # calibração estéreo
python3 movie3d.py       # geração e apresentação da imagem 3D
```

A célula seguinte encapsula o **pipeline de calibração estéreo** (equivalente ao `calibrate.py`), de forma reutilizável tanto para as imagens do exemplo quanto para as imagens da nossa câmera. Ela detecta os cantos do tabuleiro, calibra cada câmera, executa a calibração estéreo e a retificação, e salva os mapas de remapeamento no arquivo XML.

In [ ]:
def calibrar_estereo(pathL, pathR, num_imagens, chessboard=(9, 6),
                     params_file="data/params_py.xml", mostrar=False):
    """
    Pipeline completo de calibração estéreo (baseado em calibrate.py do LearnOpenCV).

    Retorna um dicionário com as matrizes e vetores obtidos e salva os mapas de
    retificação em params_file.
    """
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # Pontos 3D do tabuleiro (Z = 0 no referencial do padrão).
    objp = np.zeros((chessboard[0] * chessboard[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:chessboard[0], 0:chessboard[1]].T.reshape(-1, 2)

    img_ptsL, img_ptsR, obj_pts = [], [], []
    imgL_gray = imgR_gray = None

    for i in range(1, num_imagens + 1):
        imgL = cv2.imread(f"{pathL}img{i}.png")
        imgR = cv2.imread(f"{pathR}img{i}.png")
        if imgL is None or imgR is None:
            continue
        imgL_gray = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)
        imgR_gray = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)

        retL, cornersL = cv2.findChessboardCorners(imgL_gray, chessboard, None)
        retR, cornersR = cv2.findChessboardCorners(imgR_gray, chessboard, None)
        if retL and retR:
            obj_pts.append(objp)
            cv2.cornerSubPix(imgL_gray, cornersL, (11, 11), (-1, -1), criteria)
            cv2.cornerSubPix(imgR_gray, cornersR, (11, 11), (-1, -1), criteria)
            img_ptsL.append(cornersL)
            img_ptsR.append(cornersR)

    if not obj_pts or imgL_gray is None:
        raise FileNotFoundError(
            f"Nenhum par de tabuleiro encontrado em {pathL} / {pathR}. "
            "Verifique se as imagens de calibração foram capturadas."
        )

    # Calibração individual de cada câmera.
    _, mtxL, distL, _, _ = cv2.calibrateCamera(obj_pts, img_ptsL, imgL_gray.shape[::-1], None, None)
    hL, wL = imgL_gray.shape[:2]
    new_mtxL, _ = cv2.getOptimalNewCameraMatrix(mtxL, distL, (wL, hL), 1, (wL, hL))

    _, mtxR, distR, _, _ = cv2.calibrateCamera(obj_pts, img_ptsR, imgR_gray.shape[::-1], None, None)
    hR, wR = imgR_gray.shape[:2]
    new_mtxR, _ = cv2.getOptimalNewCameraMatrix(mtxR, distR, (wR, hR), 1, (wR, hR))

    # Calibração estéreo (intrínsecos fixos => calcula R, t, E, F).
    flags = cv2.CALIB_FIX_INTRINSIC
    criteria_stereo = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    retS, new_mtxL, distL, new_mtxR, distR, Rot, Trns, Emat, Fmat = cv2.stereoCalibrate(
        obj_pts, img_ptsL, img_ptsR, new_mtxL, distL, new_mtxR, distR,
        imgL_gray.shape[::-1], criteria_stereo, flags)

    # Retificação estéreo e matriz de reprojeção Q.
    rect_l, rect_r, proj_l, proj_r, Q, _, _ = cv2.stereoRectify(
        new_mtxL, distL, new_mtxR, distR, imgL_gray.shape[::-1], Rot, Trns, 1, (0, 0))

    LMap = cv2.initUndistortRectifyMap(new_mtxL, distL, rect_l, proj_l, imgL_gray.shape[::-1], cv2.CV_16SC2)
    RMap = cv2.initUndistortRectifyMap(new_mtxR, distR, rect_r, proj_r, imgR_gray.shape[::-1], cv2.CV_16SC2)

    cv_file = cv2.FileStorage(params_file, cv2.FILE_STORAGE_WRITE)
    cv_file.write("Left_Stereo_Map_x", LMap[0])
    cv_file.write("Left_Stereo_Map_y", LMap[1])
    cv_file.write("Right_Stereo_Map_x", RMap[0])
    cv_file.write("Right_Stereo_Map_y", RMap[1])
    cv_file.release()

    return {
        "erro_reprojecao": retS, "mtxL": new_mtxL, "distL": distL,
        "mtxR": new_mtxR, "distR": distR, "R": Rot, "t": Trns,
        "E": Emat, "F": Fmat, "Q": Q, "n_pares": len(obj_pts),
        "params_file": params_file,
    }

Execução da calibração com as **imagens fornecidas pelo exemplo**. Para reproduzir, baixe a pasta `stereo-camera` da referência [2] e copie as subpastas `stereoL/` e `stereoR/` para `data/` (o exemplo fornece 27 pares de imagens).

> **`[ESPAÇO RESERVADO PARA MÍDIA]`** — As imagens do exemplo (`data/stereoL/img*.png` e `data/stereoR/img*.png`) devem ser baixadas do repositório LearnOpenCV. A célula abaixo executa a calibração se as imagens estiverem presentes.

In [ ]:
# Calibração com as imagens fornecidas pelo exemplo (27 pares).
if os.path.exists("data/stereoL/img1.png") and os.path.exists("data/stereoR/img1.png"):
    resultado_exemplo = calibrar_estereo(
        pathL="data/stereoL/", pathR="data/stereoR/",
        num_imagens=27, chessboard=(9, 6),
        params_file="data/params_py.xml")
    print("Calibração do exemplo concluída.")
    print(f"Pares válidos: {resultado_exemplo['n_pares']}")
    print(f"Erro de reprojeção estéreo (RMS): {resultado_exemplo['erro_reprojecao']:.4f}")
else:
    resultado_exemplo = None
    print("[ESPAÇO RESERVADO] Baixe as imagens do exemplo (LearnOpenCV) para data/stereoL e data/stereoR.")

### 3.3 Parte 3 (C) — Captura e Calibração da NOSSA Câmera Estéreo

Com a câmera construída na Parte 2, capturamos nossas próprias imagens do padrão de calibração (o mesmo tabuleiro da aula anterior) e imagens de teste. Os scripts de exemplo foram adaptados para os nossos experimentos (arquivos com o sufixo do nome de um integrante da equipe):

- `capture_images_joao.py` — captura de **10 a 15 pares** de imagens do tabuleiro. O nome do arquivo de saída foi alterado para o nome de um integrante da equipe.
- `calibrate_joao.py` — calibra a nossa câmera e salva os parâmetros em `data/params_joao.xml`.
- `movie3d_joao.py` — apresenta a imagem 3D **ao vivo** das nossas webcams.

**Valores que foram modificados nos códigos de exemplo para os nossos experimentos:**

| Parâmetro | Exemplo | Nossa câmera |
|---|---|---|
| IDs das câmeras (`CamL_id`, `CamR_id`) | 1, 2 | `[preencher]` |
| Número de pares de calibração | 27 | 10–15 |
| Dimensões do tabuleiro (`CHESSBOARD`) | (9, 6) | `[confirmar padrão usado]` |
| Fonte em `movie3d` | arquivos `.mp4` | webcams ao vivo |
| Arquivo de parâmetros | `params_py.xml` | `params_joao.xml` |

Os comandos executados no laboratório foram:

```bash
python3 capture_images_joao.py    # captura 10-15 pares do tabuleiro
python3 calibrate_joao.py         # calibra e salva data/params_joao.xml
python3 movie3d_joao.py           # imagem 3D anaglifo ao vivo
```

> **`[ESPAÇO RESERVADO PARA MÍDIA]`** — Copiar as imagens de calibração capturadas pela nossa câmera para `data/stereoL/` e `data/stereoR/` (substituindo/renomeando conforme necessário) e ajustar `NUM_IMAGENS` abaixo. A célula executa a calibração da nossa câmera.

In [ ]:
# Calibração da NOSSA câmera estéreo (ajustar NUM_IMAGENS ao número capturado).
NUM_IMAGENS = 15  # entre 10 e 15

executar = os.path.exists("data/stereoL/img1.png") and os.path.exists("data/stereoR/img1.png")
if executar:
    resultado_nossa = calibrar_estereo(
        pathL="data/stereoL/", pathR="data/stereoR/",
        num_imagens=NUM_IMAGENS, chessboard=(9, 6),
        params_file="data/params_joao.xml")
    print("Calibração da nossa câmera concluída.")
    print(f"Pares válidos: {resultado_nossa['n_pares']}")
    print(f"Erro de reprojeção estéreo (RMS): {resultado_nossa['erro_reprojecao']:.4f}")
else:
    resultado_nossa = None
    print("[ESPAÇO RESERVADO] Capture as imagens com capture_images_joao.py "
          "e coloque-as em data/stereoL e data/stereoR.")

A célula abaixo exibe os **parâmetros obtidos** para a nossa câmera estéreo na forma de matrizes e vetores. Estes são os valores solicitados no item (C) do enunciado.

In [ ]:
# Exibição dos parâmetros da nossa câmera estéreo (na forma de matrizes/vetores).
np.set_printoptions(precision=4, suppress=True)

_res = resultado_nossa if resultado_nossa is not None else resultado_exemplo
if _res is not None:
    print("Matriz intrínseca — câmera ESQUERDA (K_L):\n", _res["mtxL"], "\n")
    print("Coef. de distorção esquerda [k1 k2 p1 p2 k3]:\n", _res["distL"].ravel(), "\n")
    print("Matriz intrínseca — câmera DIREITA (K_R):\n", _res["mtxR"], "\n")
    print("Coef. de distorção direita  [k1 k2 p1 p2 k3]:\n", _res["distR"].ravel(), "\n")
    print("Rotação entre as câmeras (R):\n", _res["R"], "\n")
    print("Translação entre as câmeras (t) [baseline]:\n", _res["t"].ravel(), "\n")
    print("Matriz essencial (E):\n", _res["E"], "\n")
    print("Matriz fundamental (F):\n", _res["F"], "\n")
    print("Matriz de reprojeção (Q):\n", _res["Q"])
else:
    print("[ESPAÇO RESERVADO] Execute a calibração acima para exibir os parâmetros.")

### 3.4 Parte 3 (C) — Imagem 3D Anaglifo Ao Vivo

O script `movie3d_joao.py` lê os mapas de retificação de `data/params_joao.xml`, corrige a distorção e retifica os frames das duas webcams ao vivo e compõe a imagem **anaglifo**: os canais azul e verde vêm da câmera direita e o canal vermelho da câmera esquerda. Com o óculos 3D (lente vermelha no olho esquerdo, ciano no direito), obtém-se a percepção de profundidade.

A função abaixo reproduz, no notebook, a **composição anaglifo** a partir de um par de imagens já retificado, permitindo documentar o resultado com capturas de tela.

In [ ]:
def compor_anaglifo(left_bgr, right_bgr, tamanho=(700, 700)):
    """Compoe uma imagem anaglifo (vermelho/ciano) a partir de um par estereo.

    Canais B e G vem da imagem direita; canal R vem da imagem esquerda.
    """
    output = right_bgr.copy()
    output[:, :, 2] = left_bgr[:, :, 2]  # canal R (vermelho) <- esquerda
    return cv2.resize(output, tamanho)

# Demonstração com o par de teste da nossa câmera, se disponível.
_pL, _pR = "imagens/teste_L.png", "imagens/teste_R.png"
if os.path.exists(_pL) and os.path.exists(_pR):
    anaglifo = compor_anaglifo(cv2.imread(_pL), cv2.imread(_pR))
    plt.figure()
    plt.imshow(cv2.cvtColor(anaglifo, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Imagem 3D anaglifo (par de teste da nossa câmera)")
    plt.show()
else:
    print(f"[ESPAÇO RESERVADO] Salve um par de teste em {_pL} e {_pR} para gerar o anaglifo.")

> **`[ESPAÇO RESERVADO PARA MÍDIA]`** — Inserir aqui uma **captura de tela** da janela "3D movie" gerada por `movie3d_joao.py` ao vivo. Salvar em `imagens/anaglifo_aovivo.png`.

In [ ]:
# Exibição da captura de tela da imagem 3D ao vivo.
_ss = "imagens/anaglifo_aovivo.png"
if os.path.exists(_ss):
    plt.figure()
    plt.imshow(mpimg.imread(_ss))
    plt.axis("off")
    plt.title("Imagem 3D anaglifo ao vivo (movie3d_joao.py)")
    plt.show()
else:
    print(f"[ESPAÇO RESERVADO] Adicione a captura de tela em: {_ss}")

### 3.5 Parte 3 (D) — Gravação do Vídeo 3D

O script `movie3d_joao_gravacao.py` foi adaptado a partir de `movie3d.py` para **gravar** aproximadamente 10 a 20 segundos de vídeo anaglifo, além de exibi-lo ao vivo. O vídeo é gravado em `videos/video3d_joao.avi` e depois convertido para MP4 (conforme pedido no enunciado):

```bash
python3 movie3d_joao_gravacao.py
ffmpeg -i videos/video3d_joao.avi videos/video3d_joao.mp4
```

> **`[ESPAÇO RESERVADO PARA MÍDIA]`** — Colocar o vídeo `videos/video3d_joao.mp4` (10–20 s) na pasta `videos/`. A célula abaixo o incorpora ao notebook, se presente.

In [ ]:
# Incorporação do vídeo 3D gravado (formato MP4).
from IPython.display import Video, display

_video = "videos/video3d_joao.mp4"
if os.path.exists(_video):
    display(Video(_video, embed=True, width=700))
else:
    print(f"[ESPAÇO RESERVADO] Adicione o vídeo 3D em: {_video}")

## 4. Análise e Discussão

Nesta seção respondemos às questões propostas no enunciado do laboratório.

### 4.1 (Parte 1) Epipolos, plano epipolar, linha epipolar, matriz fundamental e disparidade

- **Epipolos:** são as projeções do centro óptico de uma câmera no plano de imagem da outra; equivalem aos pontos onde a linha de base (que une os dois centros ópticos) intercepta cada plano de imagem. Todas as linhas epipolares de uma imagem convergem para o seu epipolo.
- **Plano epipolar:** plano que contém um ponto 3D da cena e os dois centros ópticos. O conjunto de todos os planos epipolares forma um feixe em torno da linha de base.
- **Linha epipolar:** interseção do plano epipolar com o plano de imagem. É a imagem, em uma câmera, do raio óptico da outra. Impõe a **restrição epipolar**: o correspondente de um ponto está sempre sobre a linha epipolar associada, reduzindo a busca de correspondências a 1D.
- **Parâmetros da matriz fundamental $F$:** matriz $3\times3$, singular ($\det F = 0$, posto 2), definida a menos de escala, com **7 graus de liberdade**. Satisfaz $x'^\top F x = 0$ para pontos correspondentes em pixels; $F x$ é a linha epipolar direita e $F^\top x'$ a esquerda; os epipolos são o espaço nulo de $F$ ($F e_L = 0$, $F^\top e_R = 0$). Quando os intrínsecos são conhecidos, obtém-se a matriz essencial $E = K_R^\top F K_L = [t]_\times R$ (5 graus de liberdade).
- **Disparidade estéreo:** diferença horizontal $d = x_L - x_R$ da posição de um ponto entre as imagens esquerda e direita retificadas. É inversamente proporcional à profundidade: $Z = f B / d$. Grande disparidade => objeto próximo; pequena disparidade => objeto distante.

### 4.2 (Parte 3B) Parâmetros necessários para a câmera estéreo e arquivos requeridos

Consultando a teoria de calibração/correção de distorção e o exemplo executado, os parâmetros necessários para descrever completamente a câmera estéreo são:

**Parâmetros intrínsecos (por câmera):**
- Matriz da câmera $K$ = $\begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$ (distâncias focais e ponto principal).
- Coeficientes de distorção da lente: radiais $k_1, k_2, k_3$ e tangenciais $p_1, p_2$.

**Parâmetros extrínsecos (relação entre as câmeras):**
- Matriz de rotação $R$ ($3\times3$) e vetor de translação $t$ ($3\times1$, cujo módulo é o *baseline*).

**Parâmetros derivados:**
- Matriz essencial $E$ e matriz fundamental $F$.
- Matrizes de retificação ($R_1, R_2$), matrizes de projeção retificadas ($P_1, P_2$) e matriz de reprojeção $Q$.
- Mapas de remapeamento (`Left_Stereo_Map`, `Right_Stereo_Map`) que aplicam undistort + rectify.

**Arquivos necessários:** sim. É preciso (i) o **conjunto de imagens do tabuleiro** (pastas `stereoL/` e `stereoR/`) para a calibração e (ii) o **arquivo de parâmetros** (`params_py.xml` no exemplo, `params_joao.xml` para a nossa câmera), gerado por `calibrate.py`/`calibrate_joao.py` e lido por `movie3d`. Sem esse XML, `movie3d.py` não consegue retificar as imagens.

### 4.3 (Parte 3C) Parâmetros e valores obtidos — quais foram salvos no XML

Os parâmetros e valores obtidos para a nossa câmera estéreo estão listados, em forma de matrizes e vetores, na saída da célula da Seção 3.3 (matrizes $K_L$, $K_R$, coeficientes de distorção, $R$, $t$, $E$, $F$ e $Q$).

**Quais parâmetros foram efetivamente salvos no arquivo XML de calibração?**

Observando o código de `calibrate.py`/`calibrate_joao.py`, o arquivo `params_*.xml` armazena **apenas os quatro mapas de remapeamento** resultantes da retificação:

```
Left_Stereo_Map_x, Left_Stereo_Map_y,
Right_Stereo_Map_x, Right_Stereo_Map_y
```

Ou seja, o XML **não** guarda diretamente $K$, $R$, $t$, $E$, $F$ ou $Q$; ele guarda o produto final desses parâmetros na forma dos mapas de undistort+rectify, que é exatamente o que `movie3d.py` precisa para gerar a imagem 3D. Essa é uma otimização: os mapas já embutem toda a correção geométrica, evitando recalcular a retificação a cada execução.

> **`[ESPAÇO RESERVADO]`** — Após a calibração, cole aqui os valores numéricos das matrizes obtidas (copiados da saída da Seção 3.3) para registro no relatório.

### 4.4 (Parte 3D) Percepção 3D — comparação entre ao vivo e gravado

> **`[ESPAÇO RESERVADO — PREENCHER APÓS O EXPERIMENTO]`**
>
> Descrever a **percepção individual de cada integrante** da equipe ao observar a imagem 3D anaglifo com o óculos vermelho/ciano, e **comparar** o resultado **ao vivo** (`movie3d_joao.py`) com o **vídeo gravado** (`movie3d_joao_gravacao.py` convertido para MP4). Pontos sugeridos para a análise:
>
> - Intensidade da sensação de profundidade (quais objetos "saltaram" à frente / recuaram).
> - Diferenças de nitidez, fluidez e latência entre o ao vivo e o gravado.
> - Efeito de *ghosting* (fantasma) e artefatos de cor do anaglifo.
> - Influência da compressão do MP4 na qualidade percebida em relação ao ao vivo.
>
> - Integrante 1 (_nome_): ...
> - Integrante 2 (_nome_): ...
> - Integrante 3 (_nome_): ...
> - Integrante 4 (_nome_): ...

### 4.5 (Parte 3E) Aplicação da Câmera Estéreo no Trabalho Final — BlindDistance

O trabalho final do grupo é o **BlindDistance** ([repositório](https://github.com/MarioCerconvis/BlindDistance)), um sistema de tecnologia assistiva que utiliza visão computacional para auxiliar pessoas com deficiência visual a navegar pelo ambiente, combinando detecção de obstáculos (YOLOv8), mapeamento de profundidade e feedback sonoro.

A câmera estéreo construída neste laboratório tem aplicação direta no BlindDistance como **fonte alternativa (ou redundante) de profundidade**, com as seguintes justificativas técnicas:

- **Estimativa de profundidade por disparidade:** a relação $Z = f\,B/d$, calibrada neste experimento, permite estimar a distância dos obstáculos usando apenas duas webcams USB de baixo custo, sem depender exclusivamente de um sensor de profundidade dedicado (como a Orbbec Astra). Isso reduz o custo do sistema e amplia a acessibilidade da solução.

- **Redundância e robustez:** sensores de profundidade ativos (infravermelho) falham sob luz solar direta ou em superfícies muito reflexivas/absorventes. Uma câmera estéreo passiva funciona bem justamente em ambientes externos bem iluminados, servindo como **fonte complementar** de profundidade e aumentando a robustez do BlindDistance em diferentes cenários.

- **Baseline ajustável ao alcance de interesse:** como observamos que a resolução de profundidade depende do baseline $B$ e da focal $f$, é possível **dimensionar a câmera estéreo** para o intervalo de distâncias mais relevante à navegação (tipicamente 0,5–4 m), otimizando a detecção de obstáculos próximos e de desníveis (degraus/buracos).

- **Fusão com a detecção YOLOv8:** as bounding boxes detectadas pelo YOLOv8 no BlindDistance podem ser associadas ao mapa de disparidade da câmera estéreo para atribuir uma **distância métrica a cada obstáculo reconhecido**, permitindo avisos sonoros do tipo "pessoa a 2 metros à frente".

- **Retificação e geometria epipolar para eficiência:** a restrição epipolar estudada garante que a busca de correspondências (e, portanto, o cálculo do mapa de disparidade em tempo real) seja feita ao longo de linhas horizontais, o que é computacionalmente eficiente e viável em hardware embarcado — requisito importante para um dispositivo vestível.

> **`[ESPAÇO RESERVADO PARA MÍDIA]`** — Inserir aqui desenho/figura ilustrando como a câmera estéreo seria integrada ao BlindDistance (por exemplo, diagrama do fluxo par estéreo → retificação → mapa de disparidade → fusão com YOLOv8 → feedback sonoro).

## 5. Conclusões

Neste laboratório, estudamos e aplicamos os conceitos de estereoscopia e geometria epipolar e construímos uma câmera estéreo didática de baixo custo:

1. **Fundamentação teórica:** compreendemos os elementos da geometria epipolar (epipolos, plano e linhas epipolares), o significado e os parâmetros da matriz fundamental e essencial, e o conceito de disparidade estéreo e sua relação inversa com a profundidade.

2. **Construção da câmera:** montamos uma câmera estereoscópica com duas webcams USB paralelas e rigidamente fixadas, com baseline definido pela distância inter-pupilar de um integrante da equipe.

3. **Calibração estéreo:** executamos o pipeline de calibração (calibração individual, `stereoCalibrate` e `stereoRectify`), obtendo os parâmetros intrínsecos, extrínsecos, matrizes essencial/fundamental e os mapas de retificação, tanto para o exemplo fornecido quanto para a nossa câmera.

4. **Imagem 3D:** geramos a imagem 3D anaglifo ao vivo e gravamos um vídeo 3D, comparando a percepção de profundidade nas duas situações.

5. **Aplicação prática:** discutimos como a câmera estéreo pode enriquecer o Trabalho Final (BlindDistance) como fonte de profundidade de baixo custo, robusta e fundível com a detecção de obstáculos.

O experimento evidenciou, na prática, que a calibração cuidadosa e a fixação rígida das câmeras são determinantes para a qualidade da percepção 3D obtida.

## 6. Referências

1. **LearnOpenCV** — *Introduction to Epipolar Geometry and Stereo Vision*. Disponível em: <https://learnopencv.com/introduction-to-epipolar-geometry-and-stereo-vision/>. Código: <https://github.com/spmallick/learnopencv/tree/master/EpipolarGeometryAndStereoVision>.

2. **LearnOpenCV** — *Making A Low-Cost Stereo Camera Using OpenCV*. Disponível em: <https://learnopencv.com/making-a-low-cost-stereo-camera-using-opencv/>. Código: <https://github.com/spmallick/learnopencv/tree/master/stereo-camera>.

3. **LearnOpenCV** — *Understanding Lens Distortion*. Disponível em: <https://learnopencv.com/understanding-lens-distortion/>. Código: <https://github.com/spmallick/learnopencv/tree/master/UnderstandingLensDistortion>.

4. **Loop, C.; Zhang, Z.** (1999). "Computing Rectifying Homographies for Stereo Vision." *IEEE Conf. Computer Vision and Pattern Recognition (CVPR)*.

5. **LearnOpenCV** — *Geometry of Image Formation*. Disponível em: <https://learnopencv.com/geometry-of-image-formation/>.

6. **Zhang, Z.** (2000). "A Flexible New Technique for Camera Calibration." *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 22(11), pp. 1330–1334.

7. **Hartley, R.; Zisserman, A.** (2003). *Multiple View Geometry in Computer Vision*. 2nd ed. Cambridge University Press.

8. **Szeliski, R.** (2022). *Computer Vision: Algorithms and Applications*. 2nd ed. Springer.

9. **OpenCV Documentation** — *Camera Calibration and 3D Reconstruction*. Disponível em: <https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html>.

10. **CNPq.** Portaria 2664/2026 — Integridade da pesquisa científica e uso de IA Generativa. Disponível em: <http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775>.

---

**Nota sobre uso de IA Generativa (Portaria CNPq 2664/2026):** Este relatório utilizou a ferramenta de IA Generativa Devin (Cognition AI) para auxiliar na estruturação, formatação e redação do texto e dos códigos-base (adaptados dos exemplos do LearnOpenCV). Todo o conteúdo final, a construção física da câmera, a captura das imagens, os experimentos, os parâmetros obtidos e as análises foram realizados, revisados e validados pelos autores, que se responsabilizam integralmente pelo conteúdo, conforme a Portaria CNPq 2664/2026.